# Lab 3 — Trino Querying & Time Travel

So far, Spark has been our only door into the lakehouse. But the whole point of an **open table format** is that the data belongs to no engine. In this lab a second engine — **Trino**, the interactive SQL powerhouse — queries the *exact same tables* Spark wrote, with no export, no copy, no sync job.

Then we exploit Iceberg's snapshot architecture for **time travel**: someone will "accidentally" corrupt a record, and we will read the past, prove what changed, and roll it back.

In this lab you will:
1. Query Spark-written Iceberg tables from **Trino**.
2. Handle the **two SQL dialects** (Spark SQL vs Trino SQL) side by side.
3. **Time travel** by snapshot ID and by timestamp — in both engines.
4. **Roll back** a bad write with one command.

> **Prerequisites** (see `README.md`): the stack is up **with the Trino service**, and you ran `day2_starter_checkpoint.py` so `hive_prod.db.orders_lakehouse` is at its end-of-Lab-2 state (20 rows, 2 snapshots).

**How multi-engine access works:** Trino never talks to Spark. Both engines ask the **Hive Metastore** for the table's current metadata pointer, then read Parquet/Avro **directly from MinIO**. The table format is the contract.

```
   Spark ──┐                       ┌── reads/writes ──> MinIO (data + metadata)
           ├──> Hive Metastore ────┤
   Trino ──┘   (pointer lookup)    └── reads/writes ──> MinIO (data + metadata)
```

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Lab03-TrinoTimeTravel").getOrCreate()

print("Rows in orders_lakehouse:", spark.table("hive_prod.db.orders_lakehouse").count())
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM hive_prod.db.orders_lakehouse.snapshots
    ORDER BY committed_at
""").show(truncate=False)

You should see **20 rows and 2 `append` snapshots** (more snapshots if you've already been through this notebook once — that's fine). If the row count is off, run the checkpoint script from the README before continuing.

## Step 1 — Connect to Trino

Trino runs as its own container (`trino`, port 8080) with an `iceberg` catalog pointed at the *same* Hive Metastore and MinIO as Spark. We talk to it with the lightweight `trino` Python client — every query below is executed **by the Trino engine**, not Spark.

In [ ]:
import pandas as pd
from trino.dbapi import connect

trino_conn = connect(host="trino", port=8080, user="student", catalog="iceberg", schema="db")

def trino_sql(query: str) -> pd.DataFrame:
    """Run a query on Trino, return the result as a pandas DataFrame."""
    cur = trino_conn.cursor()
    cur.execute(query)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])

trino_sql("SELECT version() AS trino_version")

### The moment of truth: read Spark's table from Trino

This table was created, written, and schema-evolved entirely by Spark. Trino has never seen it before.

In [ ]:
trino_sql("""
    SELECT count(*) AS row_count,
           sum(order_amount) AS total_amount,
           count(discount_applied) AS rows_with_discount_flag
    FROM orders_lakehouse
""")

In [ ]:
# Old rows (NULL discount) and new rows read together — the Lab 2 schema
# evolution is fully visible to a completely different engine.
trino_sql("SELECT * FROM orders_lakehouse ORDER BY order_id LIMIT 20")

### Iceberg metadata tables, Trino style

Trino exposes the same metadata tables you used in Spark, but with a `$` naming convention (note the double quotes — `$` needs them):

In [ ]:
trino_sql('SELECT snapshot_id, committed_at, operation FROM "orders_lakehouse$snapshots" ORDER BY committed_at')

## Dialect cheat-sheet: one table, two engines

| Task | Spark SQL (`hive_prod` catalog) | Trino (`iceberg` catalog) |
|---|---|---|
| Read a table | `SELECT ... FROM hive_prod.db.orders_lakehouse` | `SELECT ... FROM iceberg.db.orders_lakehouse` |
| Snapshots | `FROM hive_prod.db.orders_lakehouse.snapshots` | `FROM "orders_lakehouse$snapshots"` |
| Time travel (snapshot) | `... VERSION AS OF <id>` | `... FOR VERSION AS OF <id>` |
| Time travel (time) | `... FOR SYSTEM_TIME AS OF TIMESTAMP '...'` | `... FOR TIMESTAMP AS OF TIMESTAMP '... UTC'` |
| Rollback | `CALL hive_prod.system.rollback_to_snapshot('db.t', <id>)` | `CALL iceberg.system.rollback_to_snapshot('db', 't', <id>)` |

Same storage, same metadata — different front doors. Keep this table handy; you'll context-switch twice more below. (Note even the rollback procedures differ: Spark takes `'db.table'` as one argument, Trino takes schema and table separately. Newer Trino versions are migrating to an `ALTER TABLE ... EXECUTE` form — check `SHOW FUNCTIONS`/docs for your version.)

## Step 2 — The incident 🚨

It's 17:55 on a Friday. Someone runs an ad-hoc "fix" against production and zeroes out an order amount. Before we play the villain, capture the current snapshot from the table's `.history` — in real life you'd dig it out after the fact.

In [ ]:
# The table's CURRENT snapshot = the latest entry in .history that is an
# ancestor of the current state. (Why not just the newest row in .snapshots?
# A previously rolled-back snapshot would be newer but abandoned — .history
# tracks what was actually current, .snapshots merely what exists.)
pre_incident = spark.sql("""
    SELECT snapshot_id, made_current_at
    FROM hive_prod.db.orders_lakehouse.history
    WHERE is_current_ancestor = true
    ORDER BY made_current_at DESC LIMIT 1
""").collect()[0]

PRE_SNAPSHOT_ID = pre_incident.snapshot_id
PRE_TIMESTAMP   = str(pre_incident.made_current_at)
print(f"Last good snapshot     : {PRE_SNAPSHOT_ID}")
print(f"Made current at (UTC)  : {PRE_TIMESTAMP}")

spark.sql("SELECT order_id, order_amount FROM hive_prod.db.orders_lakehouse WHERE order_id = 1").show()

In [ ]:
# The fat-fingered "fix" (note: a real UPDATE on a data lake — Iceberg makes
# this a first-class, transactional operation)
spark.sql("""
    UPDATE hive_prod.db.orders_lakehouse
    SET order_amount = 0.0
    WHERE order_id = 1
""")

spark.sql("SELECT order_id, order_amount FROM hive_prod.db.orders_lakehouse WHERE order_id = 1").show()

The damage is live: `order_id = 1` now reads **0.00** for every user, in every engine. But remember Lab 1: Iceberg never modifies files in place — the UPDATE created a **new snapshot** with a rewritten data file. The old snapshot, with the old file, is still sitting in MinIO.

## Step 3 — Time travel (Spark dialect)

Two ways to address the past: by **snapshot ID** (exact) or by **timestamp** ("as of Friday 17:00").

In [ ]:
print("— Current table state —")
spark.sql("SELECT order_id, order_amount FROM hive_prod.db.orders_lakehouse WHERE order_id = 1").show()

print(f"— VERSION AS OF {PRE_SNAPSHOT_ID} —")
spark.sql(f"""
    SELECT order_id, order_amount
    FROM hive_prod.db.orders_lakehouse VERSION AS OF {PRE_SNAPSHOT_ID}
    WHERE order_id = 1
""").show()

print(f"— FOR SYSTEM_TIME AS OF '{PRE_TIMESTAMP}' —")
spark.sql(f"""
    SELECT order_id, order_amount
    FROM hive_prod.db.orders_lakehouse FOR SYSTEM_TIME AS OF TIMESTAMP '{PRE_TIMESTAMP}'
    WHERE order_id = 1
""").show()

### ✋ Verify before you continue

Look carefully at the three results above. `order_id = 1` must show:
- its **original value (49.99)** in both time-travel queries, and
- **0.00** in the standard (current) table view.

If both are true, you have just read **two versions of the same table at once** — no restore from backup, no copy. 

**Discuss:** where does the 49.99 physically live right now? (Hint: browse `data/order_timestamp_day=2026-06-28/` in MinIO — how many Parquet files are there after the UPDATE?)

## Step 4 — Time travel (Trino dialect)

Same superpower, second engine. Trino says `FOR TIMESTAMP AS OF` / `FOR VERSION AS OF`, and its timestamp literal needs an explicit time zone:

In [ ]:
print("Current state, seen from Trino:")
display(trino_sql("SELECT order_id, order_amount FROM orders_lakehouse WHERE order_id = 1"))

print(f"FOR VERSION AS OF {PRE_SNAPSHOT_ID}:")
display(trino_sql(f"""
    SELECT order_id, order_amount
    FROM orders_lakehouse FOR VERSION AS OF {PRE_SNAPSHOT_ID}
    WHERE order_id = 1
"""))

print(f"FOR TIMESTAMP AS OF TIMESTAMP '{PRE_TIMESTAMP} UTC':")
display(trino_sql(f"""
    SELECT order_id, order_amount
    FROM orders_lakehouse FOR TIMESTAMP AS OF TIMESTAMP '{PRE_TIMESTAMP} UTC'
    WHERE order_id = 1
"""))

## Step 5 — Roll it back

Time travel is read-only. To actually undo the incident we point the table's *current* state back at the last good snapshot. This is a **metadata-only** operation — instant, regardless of table size — done here with Iceberg's Spark procedure:

In [ ]:
spark.sql(f"""
    CALL hive_prod.system.rollback_to_snapshot('db.orders_lakehouse', {PRE_SNAPSHOT_ID})
""").show()

print("After rollback — Spark sees:")
spark.sql("SELECT order_id, order_amount FROM hive_prod.db.orders_lakehouse WHERE order_id = 1").show()

print("After rollback — Trino sees:")
display(trino_sql("SELECT order_id, order_amount FROM orders_lakehouse WHERE order_id = 1"))

**49.99 is back — in both engines simultaneously**, because the rollback changed the one thing they both read: the catalog's metadata pointer.

The audit trail survives, too. Notice the incident snapshot is *still there* (nothing was deleted), and the rollback shows up in the history:

In [ ]:
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM hive_prod.db.orders_lakehouse.snapshots
    ORDER BY committed_at
""").show(truncate=False)

spark.sql("""
    SELECT made_current_at, snapshot_id, is_current_ancestor
    FROM hive_prod.db.orders_lakehouse.history
    ORDER BY made_current_at
""").show(truncate=False)

Reading `.history`: the UPDATE's snapshot has `is_current_ancestor = false` — it's been "orphaned" off the main timeline by the rollback, but remains queryable by ID until it is expired (that's Lab 4).

## ✅ Wrap-up

You have:
- ✔ Queried Spark-written Iceberg tables from **Trino** with zero data movement
- ✔ Switched between the two SQL dialects (keep the cheat-sheet!)
- ✔ Time traveled by snapshot ID and timestamp in **both** engines
- ✔ Rolled back a bad write instantly, with the audit trail intact

**Optional experiments:**
- Open the Trino UI at <http://localhost:8080/ui/> (any username) and inspect your queries' execution plans.
- Try writing from Trino: `INSERT INTO orders_lakehouse VALUES (...)` via `trino_sql` — then read it back from Spark.
- Time travel to *before* the schema evolution (`VERSION AS OF` the first snapshot) — what columns come back?

**Next (Lab 4):** all those snapshots and small files pile up — lakehouse **maintenance**: compaction, snapshot expiry, and orphan cleanup.